In [7]:
import sys
sys.path.append('.')
from reconstruct_network import build_musina_network
import pandapower as pp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
net = build_musina_network()
pp.runpp(net, algorithm='nr', calculate_voltage_angles=True)
print('Load flow converged: True')

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


Load flow converged: True


In [9]:
bus_results = net.res_bus.copy()
bus_results['name'] = net.bus['name']
print(bus_results[['name', 'vm_pu', 'va_degree', 'p_mw', 'q_mvar']].round(3))

                 name  vm_pu  va_degree   p_mw  q_mvar
0    Bus_33kV_Incomer  1.020      0.000 -0.754  -0.389
1       Bus_11kV_Main  1.018     29.811  0.000   0.000
2     Bus_F1_Hospital  1.014     29.765  0.180   0.072
3          Bus_F2_CBD  1.015     29.772  0.210   0.084
4  Bus_F3_Residential  1.013     29.752  0.140   0.056
5   Bus_F4_Industrial  1.015     29.777  0.195   0.078


In [10]:
line_results = net.res_line.copy()
line_results['name'] = net.line['name']
print(line_results[['name', 'loading_percent', 'pl_mw', 'ql_mvar']].round(2))

trafo_results = net.res_trafo.copy()
trafo_results['name'] = net.trafo['name']
print('\nTransformer loading (%):')
print(trafo_results[['name', 'loading_percent']].round(1))

                  name  loading_percent  pl_mw  ql_mvar
0     Line_F1_Hospital             3.34    0.0     -0.0
1          Line_F2_CBD             3.90    0.0     -0.0
2  Line_F3_Residential             2.60    0.0     -0.0
3   Line_F4_Industrial             3.62    0.0     -0.0

Transformer loading (%):
             name  loading_percent
0  T1_20MVA_Dyn11              4.2


In [11]:
I_load_primary = 210   # A
plug_setting_secondary = 1.25  # A (125% of 1A tap)

CT_ratio_minimum = 1.2 * I_load_primary / plug_setting_secondary
print(f"Minimum CT ratio to avoid pickup on load: {CT_ratio_minimum:.0f}/1")

print("\nCandidate CT ratios and load margin:")
I_fault_11kv_max = 8500  # A (from later fault study, anticipated)
for ct in [200, 300, 400]:
    I_load_sec = I_load_primary / ct
    I_fault_sec = I_fault_11kv_max / ct
    margin = (plug_setting_secondary - 1.2 * I_load_sec) / plug_setting_secondary * 100
    print(f"CT {ct}/1: load secondary={I_load_sec:.2f} A, "
          f"fault secondary={I_fault_sec:.1f} A, load margin={margin:.1f}%")

print("\nConclusion: 300/1 provides comfortable margin and matches other bays. "
      "200/1 is marginal. 400/1 oversizes, reducing sensitivity. "
      "Most probable ratio: 300/1. Verification by injection test required.")

Minimum CT ratio to avoid pickup on load: 202/1

Candidate CT ratios and load margin:
CT 200/1: load secondary=1.05 A, fault secondary=42.5 A, load margin=-0.8%
CT 300/1: load secondary=0.70 A, fault secondary=28.3 A, load margin=32.8%
CT 400/1: load secondary=0.53 A, fault secondary=21.2 A, load margin=49.6%

Conclusion: 300/1 provides comfortable margin and matches other bays. 200/1 is marginal. 400/1 oversizes, reducing sensitivity. Most probable ratio: 300/1. Verification by injection test required.


In [12]:
f3_bus = net.bus[net.bus['name'] == 'Bus_F3_Residential'].index[0]
f3_v_pu = net.res_bus.at[f3_bus, 'vm_pu']
print(f"F3 (residential, 6.8 km) voltage: {f3_v_pu:.4f} pu")
if f3_v_pu < 0.95:
    print("Low voltage warning: exceeds 0.95 pu threshold.")

F3 (residential, 6.8 km) voltage: 1.0129 pu
